In [4]:
#r "nuget: Microsoft.Data.Analysis, 0.22.2"
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
#r "nuget: Plotly.NET.ImageExport, 6.1.0"
#r "nuget: Combinatorics, 2.0.0"

Installed Packages Combinatorics, 2.0.0 Microsoft.Data.Analysis, 0.22.2 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.ImageExport, 6.1.0 Plotly.NET.Interactive, 5.0.0

In [5]:
using System.Collections.Frozen;
using Microsoft.Data.Analysis;

In [6]:
IReadOnlyDictionary<string, string> platform;
IReadOnlyDictionary<string, int> test_case_ID;
{
    Dictionary<string, string> _platform = new Dictionary<string, string>(); 
    Dictionary<string, int> _test_case_ID = new Dictionary<string, int>();
    {
        DataFrame df = DataFrame.LoadCsv("log-prop.csv", dataTypes: Enumerable.Repeat(typeof(string), 5).ToArray());
        foreach (var row_seg in df.Rows) { 
            if (string.IsNullOrEmpty(row_seg[df.Columns.IndexOf("prefix")].ToString()) == false) {
                _platform.Add(row_seg[df.Columns.IndexOf("prefix")].ToString(), row_seg[df.Columns.IndexOf("platform")].ToString());
                _test_case_ID.Add(row_seg[df.Columns.IndexOf("prefix")].ToString(), int.Parse(row_seg[df.Columns.IndexOf("test case ID")].ToString()));
            }
        }
    }
    platform = _platform.ToFrozenDictionary();
    test_case_ID = _test_case_ID.ToFrozenDictionary();
}
var data_types = new Dictionary<string, Type[]>(StringComparer.OrdinalIgnoreCase) {
    { "lab.original", new Type[] { typeof(DateTime) }.Concat(Enumerable.Repeat(typeof(double), 4)).ToArray() },
    { "lab.vreapi", new Type[] { typeof(DateTime) }.Concat(Enumerable.Repeat(typeof(double), 8)).ToArray() },
    { "rstudio", new Type[] { typeof(DateTime) }.Concat(Enumerable.Repeat(typeof(double), 8)).ToArray() }
}.ToFrozenDictionary();

In [7]:
using System.IO;
using Plotly.NET.CSharp;

In [8]:
using container_size_t = int;

In [9]:
static class Util {
    public static readonly string[] column_prefix = new[] { "CPU:", "mem:" };
    public static readonly string[] column_suffix = new[] { "%", "Mi" };
    public static class Plot {
        public const int default_width = 1500;
        public const int default_height = 500;
        struct column_property_set_t {
            public string LineColor { get; set; }
        }
        static readonly string[] common_color = new[] { "magenta", "orangered", "limegreen", "royalblue", };
        public static Plotly.NET.GenericChart generate(DataFrame df, int width = default_width, int height = default_height) {
            var x_values = df.Columns["time"].Cast<DateTime>().ToArray();
            var CPU_util_column_property = new Dictionary<string, column_property_set_t>();
            var mem_util_column_property = new Dictionary<string, column_property_set_t>();
            var column_properties = new[] {
                CPU_util_column_property,
                mem_util_column_property
            };
            for (var i = 0; i < column_prefix.Length; i++) {
                var column_name = df.Columns.Select(c => c.Name).Where(s => s.StartsWith(column_prefix[i])).ToArray();
                for (var j = 0; j < column_name.Length; j++) {
                    column_properties[i].Add(column_name[j], new column_property_set_t { LineColor = common_color[j] });
                }
            }
            var subchart_sets = column_properties.Select(
                property_sets => property_sets.Select(
                    column_property_set => Chart.Line<DateTime, double, string>(
                        x: x_values,
                        y: df.Columns[column_property_set.Key].Cast<double>().ToArray(),
                        Name: column_property_set.Key,
                        LineColor: Plotly.NET.Color.fromString(column_property_set.Value.LineColor as string)
                    )
                ).ToArray()
            ).ToArray();
            var final_chart = Chart.Grid(
                subchart_sets.Select(subchart_set => Chart.Combine(subchart_set)).ToArray(),
                1, 2
            ).WithSize(width, height);
            return final_chart;
        }
    }
}

In [10]:
string log_path = "log";
string log_file_suffix = ".cooked.csv";
FileInfo[] util_files = new DirectoryInfo(log_path).GetFiles($"*{log_file_suffix}", SearchOption.AllDirectories);
System.IO.Directory.CreateDirectory("export/util");

In [11]:
var platforms = new[] { "lab.original", "lab.vreapi", "rstudio", };
DataFrame overall = new DataFrame(); {
    // var column_names = Util.column_prefix.SelectMany(prefix => platforms, (p1, p2) => $"avg:{p1}{p2}").ToArray();
    var column_names = Enumerable.Range(0, Util.column_prefix.Length).SelectMany(i => platforms, (i, p) => $"ave:{Util.column_prefix[i]}{p} ({Util.column_suffix[i]})").ToArray();
    // Console.WriteLine(string.Join(", ", column_names));
    var row_count = util_files.Length / platforms.Length;
    foreach (var column_name in column_names) {
        overall.Columns.Add(new PrimitiveDataFrameColumn<double>(column_name, row_count));
    }
}
var number_of_columns_to_sum = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase) {
    { "lab.original", 2 },
    { "lab.vreapi", 4 },
    { "rstudio", 4 }
}.ToFrozenDictionary();

In [12]:
using Plotly.NET.ImageExport;

In [13]:
Dictionary<string, container_size_t> current_row = platforms.ToDictionary(e => e, _ => 0, StringComparer.OrdinalIgnoreCase);
foreach (var util_file in util_files) {
    //Console.WriteLine($"﷿﷿﷿﷿﷿﷿Processing {util_file}");
    var prefix = util_file.Name.Substring(0, util_file.Name.Length - log_file_suffix.Length);
    var platform_of_current_file = platform[prefix];
    var test_case_ID_of_current_file = test_case_ID[prefix];
    DataFrame df = DataFrame.LoadCsv(util_file.FullName, dataTypes: data_types[platform_of_current_file]);
    {
        double CPU_util = 0;
        double mem_util = 0;
        for (var i = 1; i <= number_of_columns_to_sum[platform_of_current_file]; i++) {
            CPU_util += (df.Columns[i] as PrimitiveDataFrameColumn<double>).Mean();
            mem_util += (df.Columns[i + number_of_columns_to_sum[platform_of_current_file]] as PrimitiveDataFrameColumn<double>).Mean();
        }
        overall.Columns[$"ave:CPU:{platform_of_current_file} (%)"][current_row[platform_of_current_file]] = CPU_util;
        overall.Columns[$"ave:mem:{platform_of_current_file} (Mi)"][current_row[platform_of_current_file]] = mem_util;
        ++current_row[platform_of_current_file];
        //Console.WriteLine($"CPU: {CPU_util}, mem: {mem_util}");
    }
    //Console.WriteLine(overall);
    //Console.WriteLine(df);
    var chart = Util.Plot.generate(df);
    display(chart);
    chart.SavePNG($"export/util/{test_case_ID_of_current_file}.{platform_of_current_file}", Width: Util.Plot.default_width, Height: Util.Plot.default_height);
}

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

...

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

In [ ]:
Console.WriteLine(overall);

ave:CPU:lab.original (%)  ave:CPU:lab.vreapi (%)    ave:CPU:rstudio (%)       ave:mem:lab.original (Mi) ave:mem:lab.vreapi (Mi)   ave:mem:rstudio (Mi)      
144.1364095744681         143.34541237113388        49.66561797752811         2424.511884973404         2735.3868194265465        1855.645299332865         
133.79820338983052        138.18800653594775        55.92754385964911         2248.4586334745763        2600.9737413194443        1862.3403645833332        
133.33620938628155        140.54300380228136        60.54717842323651         2194.119359205776         2588.70423894962          1862.1979382780085        
143.55084112149524        141.47677419354855        48.99591304347821         2496.724402501947         2868.005418346774         1888.1995516304348        
117.5973333333333         121.7031578947368         85.79222222222225         2049.036197916667         2458.6711554276317        1827.2033781828702        
150.99210873146626        140.57206429780027        48.023

In [15]:
using Combinatorics.Collections;

In [16]:
var combos = new Combinations<container_size_t>(Enumerable.Range(0, platforms.Length), 2);
foreach (var (item, index) in new string[] { "CPU", "mem", }.Select((value, index) => (value, index))) {
    foreach (var combo in combos) {
        var @ref = (PrimitiveDataFrameColumn<double>)overall.Columns[$"ave:{item}:{platforms[combo[0]]} ({Util.column_suffix[index]})"];
        var diff = (PrimitiveDataFrameColumn<double>)overall.Columns[$"ave:{item}:{platforms[combo[1]]} ({Util.column_suffix[index]})"];
        var rate = (PrimitiveDataFrameColumn<double>)(diff - @ref) / @ref * 100.0;
        Console.WriteLine($"{diff.Name} v. {@ref.Name} (%)");
        // foreach (var r in rate) { Console.WriteLine(r); }
        Console.WriteLine($"Ave: {rate.Mean()}{Environment.NewLine}Med: {rate.Median()}{Environment.NewLine}Max: {rate.Max()}{Environment.NewLine}");
    }
}

ave:CPU:lab.vreapi (%) v. ave:CPU:lab.original (%) (%)
Ave: 0.5101919519593318
Med: 1.2595447004426819
Max: 5.404979224451605

ave:CPU:rstudio (%) v. ave:CPU:lab.original (%) (%)
Ave: -59.12995175999588
Med: -63.8387281513539
Max: -27.04577579234596

ave:CPU:rstudio (%) v. ave:CPU:lab.vreapi (%) (%)
Ave: -59.52622201823533
Med: -65.25128728105575
Max: -29.506987570178378

ave:mem:lab.vreapi (Mi) v. ave:mem:lab.original (Mi) (%)
Ave: 14.324082720629528
Med: 13.71383753157238
Max: 19.991592043491288

ave:mem:rstudio (Mi) v. ave:mem:lab.original (Mi) (%)
Ave: -21.095408576652066
Med: -23.463138669942186
Max: -10.826203068513019

ave:mem:rstudio (Mi) v. ave:mem:lab.vreapi (Mi) (%)
Ave: -31.043894642518563
Med: -32.16150322308391
Max: -25.683295460263842

